# Classificação

Nesse notebook vamos explorar alguns modelos de classificação de imagens que vimos em aula.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que usamos em outros notebooks, vamos utilizar mais algumas.
* `pathlib`: Para trabalhar com pastas/ diretórios.
* `urllib`: Para download de arquivos.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image


## 1. Introdução ao Módulo DNN do OpenCV

O módulo **DNN (Deep Neural Network)** do OpenCV permite executar modelos de aprendizado profundo diretamente na biblioteca, sem depender de frameworks como PyTorch ou TensorFlow durante a inferência. Seu objetivo é facilitar a integração entre técnicas clássicas de visão computacional e modelos modernos de Inteligência Artificial.
Com o OpenCV DNN é possível utilizar modelos pré-treinados para diversas tarefas, como:

* Classificação de imagens
* Detecção de objetos
* Segmentação de imagens
* Reconhecimento facial
* OCR (Reconhecimento Óptico de Caracteres)
* Estimativa de pose humana

O módulo suporta modelos provenientes de diferentes ecossistemas, incluindo **ONNX**, TensorFlow, Caffe, Darknet (YOLO) e outros formatos amplamente utilizados pela comunidade. Atualmente, o formato **ONNX** tornou-se a principal opção por permitir a interoperabilidade entre diferentes frameworks.
Uma das principais vantagens do OpenCV DNN é sua simplicidade. O fluxo básico de inferência consiste em:

1. Carregar o modelo.
2. Preparar a imagem de entrada.
3. Converter a imagem para um blob.
4. Executar a inferência.
5. Interpretar os resultados.

### 1.1. O que é ONNX?

**ONNX (Open Neural Network Exchange)** é um formato padrão para armazenar e compartilhar modelos de Inteligência Artificial. Seu principal objetivo é permitir que um modelo treinado em um framework possa ser utilizado em outro ambiente sem precisar ser reescrito. 

Antes do ONNX, era comum ficar preso ao framework utilizado durante o treinamento. Um modelo treinado em PyTorch normalmente precisava ser executado em PyTorch; um modelo treinado em TensorFlow dependia do TensorFlow. O ONNX surgiu para resolver esse problema de interoperabilidade.

#### O que existe dentro de um arquivo ONNX?
Um arquivo **.onnx** contém:

* Arquitetura da rede;
* Camadas da rede neural;
* Pesos treinados;
* Informações de entrada e saída;
* Operadores necessários para executar o modelo.

## 2. MobileNetV2

O **MobileNetV2** é uma rede neural convolucional (CNN) desenvolvida pelo Google em 2018 com foco em **eficiência**. Seu objetivo é oferecer boa precisão em tarefas de visão computacional utilizando muito menos memória e processamento do que redes tradicionais.

### 2.1. Carregando o Modelo

Vamos baixar o modelo do github, salvá-lo na nossa pasta `modelos` e carregá-lo usando o módulo DNN do OpenCV.

#### 2.1.1. Baixando modelo na pasta local de modelos

In [ ]:
def baixar_arquivo_modelo(url: str, nome_arquivo: str) -> Path:
    # Pasta para armazenar os modelos
    MODELOS_DIR = Path('modelos')
    MODELOS_DIR.mkdir(exist_ok=True)

    path_modelo = MODELOS_DIR / nome_arquivo

    if not path_modelo.exists():
        print("Baixando arquivo...")
        urlretrieve(url, path_modelo)
        print("Download concluído!")
    else:
        print("Arquivo já existe.")

    return path_modelo

In [ ]:
url = (
    "https://github.com/onnx/models/raw/refs/heads/"
    "main/validated/vision/classification/mobilenet/model/mobilenetv2-12.onnx"
)

# baixando modelo
modelo_arquivo = baixar_arquivo_modelo(url, 'mobilenetv2-12.onnx')

#### 2.1.2. Carregando modelo no módulo DNN

In [ ]:
net_mobilenetv2 = cv2.dnn.readNet(str(modelo_arquivo))

print("Modelo carregado com sucesso!")

### 2.2. Preparando Imagem de Entrada

Vamos carregar usando o próprio OpenCV da forma que já vimos.

In [ ]:
img = cv2.imread('imagens/02/luke.jpg')

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show();

### 2.3. Convertendo uma Imagem para um Blob

Uma imagem carregada pelo OpenCV é apenas uma matriz de pixels.

Por exemplo, uma foto de 224×224 pixels colorida possui o formato:
``` python
(224, 224, 3)
```

onde:

* 224 = altura
* 224 = largura
* 3 = canais de cor (BGR)

Entretanto, redes neurais não esperam receber imagens nesse formato. Elas foram treinadas para receber dados organizados de uma maneira específica, com tamanho, escala e estrutura padronizados.

#### 2.3.1. O que é um Blob?

Um **blob** (*Binary Large Object*) é uma representação numérica da imagem preparada para ser utilizada pela rede neural.

No OpenCV DNN, o blob normalmente possui quatro dimensões: `(N, C, H, W)`

onde:
* N = quantidade de imagens (batch)
* C = canais
* H = altura
* W = largura

Por exemplo:
``` python
(1, 3, 244, 244)
```

#### 2.3.2. O que acontece durante a conversão?

A função `blobFromImage()` realiza automaticamente diversas etapas de pré-processamento exigida pelos modelos.

* **Redimensionamento:** a imagem é ajustada para o tamanho esperado pela rede.
* **Normalização:** pixels geralmente variam de 0 a 255, mas muitas redes foram treinadas usando valores entre 0 e 1.
* **Troca de canais:** o OpenCV lê imagens em `BGR`, mas a maioria dos modelos foi treinada em `RGB`.
* **Subtração da média:** alguns modelos exigem que seja removida a média dos canais de cor para que a entrada fique semelhante aos dados usados durante o treinamento.

In [ ]:
def converter_blob_mobilenetv2(img: np.ndarray) -> np.ndarray:
    blob = cv2.dnn.blobFromImage(
        img,
        scalefactor=1/255.,
        size=(224, 224),
        mean=(0.485, 0.456, 0.406),
        swapRB=True,
        crop=False
    )

    # Ajuste do desvio padrão ImageNet
    blob[0, 0] /= 0.229
    blob[0, 1] /= 0.224
    blob[0, 2] /= 0.225

    return blob

In [ ]:
print("Shape img:", img.shape)

blob = converter_blob_mobilenetv2(img)
print("Shape blob:", blob.shape)

### 2.4. Executando a Inferência

Primeiro usamos o método `setInput()` para passar o blob e em seguida usamos o método `forward()` para passar o blob para rede neural. 

In [ ]:
net_mobilenetv2.setInput(blob)
output = net_mobilenetv2.forward()

print("Shape output:", output.shape)

Note que o shape do output foi `(1, 1000)`, onde:

* 1 = quantidade de imagens (igual ao que enviamos).
* 1000 = scores ou probabilidade para as 1000 categorias que o modelo foi treinado. No caso, na base `ImageNet`.

### 2.5. Interpretando os Resultados

No caso da classificação, queremos saber qual das categorias foi indicada para a imagem que enviamos dentre as 1000 possíveis e qual o score associado.

#### 2.5.1. Obtendo o ID da categoria que obteve o melhor score

In [ ]:
scores = output.flatten()
classe_id = np.argmax(scores)

print("Classe:", classe_id)
print("Confiança:", scores[classe_id])

#### 2.5.2. Obtendo a descrição da categoria

In [ ]:
url = (
    "https://github.com/onnx/models/raw/refs/heads/"
    "main/validated/vision/classification/synset.txt"
)

# baixando arquivo com as classes
classes_arquivo = baixar_arquivo_modelo(url, 'synset.txt')

In [ ]:
with open(classes_arquivo, 'r') as f:
    classes = [linha.strip().split(maxsplit=1)[-1] for linha in f]

print("Linhas do arquivo:", len(classes))

In [ ]:
print("Classe:", classe_id)
print("Descrição:", classes[classe_id])
print("Confiança:", scores[classe_id])

### 2.6. Testando em uma Outra Imagem

In [ ]:
img = cv2.imread('imagens/02/guitarra.jpg')

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
def inferir_mobilenetv2(img: np.ndarray) -> None:
    # Convertendo a imagem em blob
    blob = converter_blob_mobilenetv2(img)

    # Inferindo usando o modelo
    net_mobilenetv2.setInput(blob)
    output = net_mobilenetv2.forward()

    # Obtendo categoria com melhor score
    scores = output.flatten()
    classe_id = np.argmax(scores)

    # Apresentando inferência
    print("Classe:", classe_id)
    print("Descrição:", classes[classe_id])
    print("Confiança:", scores[classe_id])

In [ ]:
# Executando inferência
inferir_mobilenetv2(img)

## 3. EfficientNet

O **EfficientNet** é uma família de redes neurais convolucionais apresentada pelo Google em 2019 com o objetivo de maximizar a precisão utilizando a menor quantidade possível de recursos computacionais.

### 3.1. Baixando, Carregando Modelo e Definindo Conversão para Blob

O processo é semelhante ao **MobileNetV2**. Vamos baixar o modelo, carregar usando o módulo DNN e criando as funções específicas para ele.

In [ ]:
url = (
    "https://github.com/onnx/models/raw/refs/heads/"
    "main/validated/vision/classification/efficientnet-lite4/model/efficientnet-lite4-11.onnx"
)

# baixando modelo
modelo_arquivo = baixar_arquivo_modelo(url, 'efficientnet-lite4-11.onnx')

In [ ]:
net_efficientnet = cv2.dnn.readNet(str(modelo_arquivo))

print("Modelo carregado com sucesso!")

In [ ]:
def converter_blob_efficientnet(img: np.ndarray) -> np.ndarray:
    # BGR -> RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Redimensiona para entrada da rede
    img_rgb = cv2.resize(img_rgb, (224, 224))

    # Converte para float
    img_rgb = img_rgb.astype(np.float32)

    # Escala para [0,1]
    img_rgb /= 255.0

    # NHWC
    blob = np.expand_dims(img_rgb, axis=0)

    return blob

In [ ]:
def inferir_efficientnet(img: np.ndarray) -> None:
    # Convertendo a imagem em blob
    blob = converter_blob_efficientnet(img)

    # Inferindo usando o modelo
    net_efficientnet.setInput(blob)
    output = net_efficientnet.forward()

    # Obtendo categoria com melhor score
    scores = output.flatten()
    classe_id = np.argmax(scores)

    # Apresentando inferência
    print("Classe:", classe_id)
    print("Descrição:", classes[classe_id])
    print("Confiança:", scores[classe_id])

### 3.2. Testando com as mesmas Imagens

In [ ]:
img = cv2.imread('imagens/02/luke.jpg')

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show();

inferir_efficientnet(img)

In [ ]:
img = cv2.imread('imagens/02/guitarra.jpg')

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show();

inferir_efficientnet(img)